# Load Packages

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from tqdm.auto import tqdm

# Import Functions
sys.path.append("../../")
from src.configs.octmnist_config import data_name, data_name_oods, batch_size, eval_batch_size, \
    seed_interval_size, ensemble_size

from src.file_manager.filepath import FilePath

from src.training.train import train_model_w_best_param
from src.training.misc import get_class_weights
from src.file_manager.load_save_model import load_model
from src.evaluation.inference import get_all_predictions
from src.file_manager.load_save_df import save_pred_df, load_pred_df, save_pred_perf_df

from src.evaluation.evaluate import get_model_performance
from src.data_generator.oct_mnist import load_octmnist_data_dict
from src.data_generator.chest_mnist import load_chestmnist_data_dict
from src.data_generator.octdl import load_octdl_data_dict
from src.data_processing.ood_dataset_preprocessing import process_dataset_for_ood, left_join_datasets
from src.models.resnet.model import ResNet18
from src.models.resnet.train import train_resnet
from src.models.de.train import train_ensemble_w_best_param
from src.models.de.predict import get_all_ensemble_predictions
from src.models.de.model import DeepEnsemble
from src.data_processing.dataloader import get_pytorch_split_dict_image
from src.models.resnet.predict import get_resnet_predictions_speedup

from cur_seed import seed
# seed = 2024

fp = FilePath(data_name=data_name, seed=seed)
fp_ood = FilePath(data_name=data_name_oods[0], seed=seed)
fp_ood2 = FilePath(data_name=data_name_oods[1], seed=seed)

# Get Data

In [ ]:
data_dict = load_octmnist_data_dict(fp_preprocessed=fp.get_preprocessed_folder())
num_ori_test = len(data_dict["test_df"])
data_dict_ood = load_chestmnist_data_dict(fp_preprocessed=fp_ood.get_preprocessed_folder(), only_test=True)
data_dict_ood = process_dataset_for_ood(data_dict, data_dict_ood, seed)
octdl_in_data_dict, octdl_out_data_dict = load_octdl_data_dict(fp_preprocessed=fp_ood2.get_preprocessed_folder())
data_dict = left_join_datasets(data_dict, octdl_in_data_dict)
octdl_out_data_dict = process_dataset_for_ood(data_dict, octdl_out_data_dict, seed)
octdl_in_data_dict = process_dataset_for_ood(data_dict, octdl_in_data_dict, seed)

# Training

In [ ]:
class_weights = get_class_weights(data_dict)
params = dict(
    ModelClass=ResNet18,
    data_dict=data_dict,
    batch_size=batch_size,
    eval_batch_size=eval_batch_size,
    train_model_func=train_resnet,
    metric_to_monitor="ce loss",
    maximise=False,
    seed=seed,
    train_param_dict = dict(
        max_epochs=500, weight_decay=0.001, patience=5, lr=0.001, class_weights=class_weights),
    pytorch_split_dict_func=get_pytorch_split_dict_image
)
train_ensemble_w_best_param(
    **params,
    best_param={},
    cur_model_name="tuned",
    seed_interval_size=seed_interval_size, ensemble_size=ensemble_size,
    data_name=data_name
)

# Prediction

In [ ]:
pred_df = get_all_ensemble_predictions(
    data_dict, pred_func=get_resnet_predictions_speedup, 
    batch_size=batch_size, eval_batch_size=eval_batch_size, 
    data_name=data_name, seed=seed, 
    ModelClass=ResNet18, cur_model_name="tuned",
    seed_interval_size=100, ensemble_size=5,
    additional_pred_args={"mc":False},
    model_label="resnet"
)
pred_df["split_perf"] = pred_df["split"]
pred_df["split_perf"][pred_df["split"]=="Test"] = ["Test-OCTMNIST" for i in range(num_ori_test)] + \
    ["Test-OCTDL" for i in range((pred_df["split"]=="Test").sum()-num_ori_test)]
save_pred_df(pred_df=pred_df, fp=fp, ModelClass=DeepEnsemble)

# Performance Evaluation

In [ ]:
pred_df = load_pred_df(fp=fp, ModelClass=DeepEnsemble)
perf_df = get_model_performance(
    all_pred_df=pred_df, data_dict=data_dict, label="de", perf_split_col="split_perf")
save_pred_perf_df(pred_perf_df=perf_df, fp=fp, ModelClass=DeepEnsemble)
perf_df

# OOD Prediction

In [ ]:
ood_dicts = {
    "ood_in_octdl": octdl_in_data_dict, 
    "ood_out_octdl": octdl_out_data_dict,
    "ood_chestmnist": data_dict_ood}
for label, cur_ood_data_dict in tqdm(ood_dicts.items(), total=len(ood_dicts)):
    pred_df_ood = get_all_ensemble_predictions(
        cur_ood_data_dict, pred_func=get_resnet_predictions_speedup, 
        batch_size=batch_size, eval_batch_size=eval_batch_size, 
        data_name=data_name, seed=seed, 
        ModelClass=ResNet18, cur_model_name="tuned",
        seed_interval_size=100, ensemble_size=5,
        additional_pred_args={"mc":False},
        model_label="resnet",
        optional_label=label
    )
    save_pred_df(pred_df=pred_df_ood, fp=fp, ModelClass=DeepEnsemble, optional_label=label)